# Whisper STT Evaluation - All Models

Evaluation of 8 Speech-to-Text models on a domain-specific dataset of 30 phrases from the **VisionLeaderUFV** campus navigation system.

All models run **locally on Colab GPU** to guarantee a fair head-to-head comparison on identical hardware. This isolates the model as the only experimental variable and removes the network confound that mixing cloud/local inference would introduce.

Each model is tested in **two conditions**:
1. **No prompt** - standard transcription with no domain context
2. **With prompt** - a domain vocabulary hint aligned with the test dataset

HuggingFace ASR models (Wav2Vec2, Speech2Text) do not accept a prompt parameter, so for them the same transcription is reported under both conditions (delta = 0 by design).

## Models

| Model | Family | Params | Backend |
|---|---|---|---|
| `whisper-tiny` | Whisper | 39M | Colab GPU |
| `whisper-base` | Whisper | 74M | Colab GPU |
| `whisper-small` | Whisper | 244M | Colab GPU |
| `whisper-medium` | Whisper | 769M | Colab GPU |
| `whisper-large-v3-turbo` | Whisper | 809M | Colab GPU |
| `whisper-large-v3` | Whisper | 1550M | Colab GPU |
| `facebook/wav2vec2-base-960h` | Wav2Vec2 (CTC) | 95M | Colab GPU |
| `facebook/s2t-small-librispeech-asr` | Speech2Text (Seq2Seq) | 30M | Colab GPU |

## Metrics

| Metric | Description | Direction |
|---|---|---|
| **WER** | Word Error Rate | Lower better |
| **CER** | Character Error Rate | Lower better |
| **Sentence Accuracy** | Fraction of phrases transcribed perfectly | Higher better |
| **DKA** | Domain Keyword Accuracy | Higher better |
| **Latency** | Avg seconds per inference on Colab GPU | Lower better |
| **PIS** | Prompt Impact Score (= DKA_with - DKA_no) | Higher = more prompt-dependent |

## Outputs

- Per-model `results.csv` with row-level transcriptions and metrics
- `all_models_summary_long.csv` (one row per model x condition)
- `all_models_summary_wide.csv` (one row per model with `no_prompt` vs `with_prompt` deltas)
- `per_utterance_predictions.csv` (one row per utterance with each model x condition prediction)
- `prompt_impact_score.csv` (models ranked by PIS)
- `utterance_difficulty.csv` (utterances ranked by average difficulty)
- Comparison plots and per-category heatmaps in `comparison_plots/`
- `whisper_reproducibility.json` with environment metadata

---
## 1. Setup

Mount Google Drive, install dependencies, and verify that the GPU is available.

### 1.1 Mount Google Drive and verify audios

Audios must be uploaded to `Drive/MyDrive/experimento_whisper/` as `utterance_01.wav` ... `utterance_20.wav`. The cell fails fast if any audio is missing.

In [ ]:
from google.colab import drive
import os
import re

drive.mount('/content/drive')

AUDIO_DIR = '/content/drive/MyDrive/experimento_whisper'
AUDIO_EXTS = ('.wav', '.m4a', '.mp3', '.flac', '.ogg', '.webm')
EXPECTED_STEMS = [f'utterance_{i:02d}' for i in range(1, 31)]

if not os.path.isdir(AUDIO_DIR):
    msg = f'Audio directory not found: {AUDIO_DIR}'
    msg += ' (create it in Drive and upload utterance_01 ... utterance_30 in any audio format)'
    raise FileNotFoundError(msg)

# Tolerant detection: accept any common audio extension and find which utterances are present.
# Builds a stem -> filename mapping (e.g. 'utterance_03' -> 'utterance_03.m4a').
_STEM_RE = re.compile(r'^(utterance_\d{2})\.([a-z0-9]+)$', re.IGNORECASE)
AUDIO_FILE_BY_STEM = {}
for fname in sorted(os.listdir(AUDIO_DIR)):
    m = _STEM_RE.match(fname)
    if not m:
        continue
    stem = m.group(1).lower()
    ext = m.group(2).lower()
    if f'.{ext}' not in AUDIO_EXTS:
        continue
    if stem in AUDIO_FILE_BY_STEM:
        prev = AUDIO_FILE_BY_STEM[stem]
        print(f'Warning: duplicate audio for {stem} -> keeping {prev}, ignoring {fname}')
        continue
    AUDIO_FILE_BY_STEM[stem] = fname

AVAILABLE_STEMS = sorted(AUDIO_FILE_BY_STEM.keys())
MISSING_STEMS = [s for s in EXPECTED_STEMS if s not in AUDIO_FILE_BY_STEM]

if not AVAILABLE_STEMS:
    raise FileNotFoundError(
        f'No utterances found in {AUDIO_DIR}. Expected files like utterance_01.wav or utterance_01.m4a'
    )

print(f'Utterances available: {len(AVAILABLE_STEMS)} / {len(EXPECTED_STEMS)}')
for stem in AVAILABLE_STEMS:
    print(f'  [OK] {AUDIO_FILE_BY_STEM[stem]}')

if MISSING_STEMS:
    print()
    print(f'Warning: {len(MISSING_STEMS)} utterances missing (will be skipped):')
    for stem in MISSING_STEMS:
        print(f'  [--] {stem}.<ext>')
    print('Re-record and re-run when ready.')


### 1.2 Install dependencies

- `ffmpeg` - required by `openai-whisper` to load audio files
- `jiwer` - WER and CER computation
- `openai-whisper` - local Whisper inference
- `librosa` + `sentencepiece` - required by the HuggingFace ASR models

**Note:** we deliberately do NOT install `torch` because Colab provides it pre-installed with CUDA support. Reinstalling via `pip` would replace it with the CPU-only version.

In [ ]:
!apt-get install -y ffmpeg -qq > /dev/null
!pip install -q jiwer openai-whisper librosa sentencepiece


### 1.3 Imports and device check

Verifies that CUDA is available. If `Device: cpu` appears, switch the runtime type to GPU before continuing.

In [ ]:
import json
import re
import time
import gc
import datetime
import platform
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import whisper
from jiwer import wer as wer_fn, cer as cer_fn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Torch: {torch.__version__}')


---
## 2. Configuration

Defines the dataset, the domain keywords, the models to evaluate, and the domain prompt used in the *with-prompt* condition.

### 2.1 Dataset - 30 phrases organized in 6 categories

Phrases are designed to cover the full real-use scope of the VisionLeaderUFV application:

| Category | Phrases | Purpose |
|---|---|---|
| `initial_location` | 5 | User describes where they are (with room codes) |
| `destination` | 5 | User states where they want to go |
| `navigation_command` | 5 | Commands during navigation (cancel, repeat, etc.) |
| `confirmation` | 5 | Yes/no/done responses during the flow |
| `hard_domain` | 5 | Long phrases combining several domain terms (stress test) |
| `edge_case` | 5 | Atypical syntax (open questions, polite forms, paraphrases) |


In [ ]:
PHRASES_ALL = [
    {'id': 'utterance_01', 'category': 'initial_location', 'ground_truth': 'I am in classroom 0.3 building G'},
    {'id': 'utterance_02', 'category': 'initial_location', 'ground_truth': 'I am in classroom 2.1 on the second floor'},
    {'id': 'utterance_03', 'category': 'initial_location', 'ground_truth': 'I am at the main entrance of building G'},
    {'id': 'utterance_04', 'category': 'initial_location', 'ground_truth': 'I am in the AI laboratory on the ground floor of building G'},
    {'id': 'utterance_05', 'category': 'initial_location', 'ground_truth': 'I am next to the elevator on floor 1 of building G'},
    {'id': 'utterance_06', 'category': 'destination', 'ground_truth': 'I want to go to classroom 1.4'},
    {'id': 'utterance_07', 'category': 'destination', 'ground_truth': 'Take me to the gym in building CD'},
    {'id': 'utterance_08', 'category': 'destination', 'ground_truth': 'I want to go to the cafeteria'},
    {'id': 'utterance_09', 'category': 'destination', 'ground_truth': 'Take me to the nearest restroom on this floor'},
    {'id': 'utterance_10', 'category': 'destination', 'ground_truth': 'Could you guide me to the EPS faculty offices please'},
    {'id': 'utterance_11', 'category': 'navigation_command', 'ground_truth': 'I am in a hurry please use the stairs'},
    {'id': 'utterance_12', 'category': 'navigation_command', 'ground_truth': 'Repeat the instructions please'},
    {'id': 'utterance_13', 'category': 'navigation_command', 'ground_truth': 'I am lost I need help'},
    {'id': 'utterance_14', 'category': 'navigation_command', 'ground_truth': 'Cancel navigation and start again'},
    {'id': 'utterance_15', 'category': 'navigation_command', 'ground_truth': 'Stop and tell me where I am'},
    {'id': 'utterance_16', 'category': 'confirmation', 'ground_truth': 'Yes I am ready to continue'},
    {'id': 'utterance_17', 'category': 'confirmation', 'ground_truth': 'No that is not my location'},
    {'id': 'utterance_18', 'category': 'confirmation', 'ground_truth': 'I have arrived at the elevator on floor 2'},
    {'id': 'utterance_19', 'category': 'confirmation', 'ground_truth': 'Done I have reached my destination'},
    {'id': 'utterance_20', 'category': 'confirmation', 'ground_truth': 'Yes I can see the elevator and the doors are open'},
    {'id': 'utterance_21', 'category': 'hard_domain', 'ground_truth': 'I am in classroom 1.1 and I want to go to the EPS faculty offices'},
    {'id': 'utterance_22', 'category': 'hard_domain', 'ground_truth': 'Take me from classroom 0.1 to the AI laboratory on the ground floor'},
    {'id': 'utterance_23', 'category': 'hard_domain', 'ground_truth': 'I just came out of the elevator on the first floor of building G'},
    {'id': 'utterance_24', 'category': 'hard_domain', 'ground_truth': 'I do not know where I am I need help finding my location'},
    {'id': 'utterance_25', 'category': 'hard_domain', 'ground_truth': 'I am at the directory sign in building G next to the mentoring room'},
    {'id': 'utterance_26', 'category': 'edge_case', 'ground_truth': 'How do I get from building CD to building G'},
    {'id': 'utterance_27', 'category': 'edge_case', 'ground_truth': 'Where is the secretary office in building G'},
    {'id': 'utterance_28', 'category': 'edge_case', 'ground_truth': 'I need to find classroom 0.3 quickly'},
    {'id': 'utterance_29', 'category': 'edge_case', 'ground_truth': 'Could you tell me how to reach the AI laboratory from here'},
    {'id': 'utterance_30', 'category': 'edge_case', 'ground_truth': 'My destination is classroom 1.4 in building G please'},
]

# Filter to only the phrases whose audio file is actually available,
# attaching the resolved filename so the inference loop works regardless of extension.
PHRASES = []
for p in PHRASES_ALL:
    fname = AUDIO_FILE_BY_STEM.get(p['id'])
    if fname is None:
        continue
    PHRASES.append({**p, 'audio': fname})

print(f'Loaded {len(PHRASES)} phrases (out of {len(PHRASES_ALL)} expected) across {len(set(p["category"] for p in PHRASES))} categories')


### 2.2 Domain keywords for DKA

Keywords whose correct transcription is critical for the navigation system to function (room codes, building codes, navigation elements, voice commands).

In [ ]:
DOMAIN_KEYWORDS = [
    # Room numbers
    '0.1', '0.3', '1.1', '1.4', '2.1',
    # Buildings
    'building g', 'building cd',
    # Navigation elements and rooms
    'elevator', 'stairs', 'cafeteria', 'ai laboratory', 'restroom',
    'faculty offices', 'main entrance', 'mentoring room',
    'secretary', 'directory sign', 'gym',
    # Floors
    'ground floor', 'first floor', 'second floor', 'floor 1', 'floor 2',
    # Voice commands and intents
    'cancel', 'repeat', 'hurry', 'lost', 'stop', 'done', 'destination',
    'doors are open',
]

print(f'Tracking {len(DOMAIN_KEYWORDS)} domain keywords')


### 2.3 Domain prompt

The same prompt the production app feeds to Whisper. It primes the model with the most likely vocabulary for this domain. The *with-prompt* condition uses this string; the *no-prompt* condition runs without any prompt context.

In [ ]:
# Domain prompt aligned strictly with the vocabulary present in the 30 test utterances.
# This isolates the experimental variable (prompt impact) by avoiding extra terms
# that don't appear in the dataset, which could otherwise bias the model in the
# with_prompt condition.
DOMAIN_PROMPT = (
    'UFV campus navigation assistant. '
    'classroom 0.1, classroom 0.3, classroom 1.1, classroom 1.4, classroom 2.1, '
    'building G, building CD, '
    'main entrance, ground floor, first floor, second floor, floor 1, floor 2, '
    'elevator, stairs, cafeteria, gym, restroom, '
    'AI laboratory, mentoring room, EPS faculty offices, secretary office, directory sign, '
    'I want to go to, take me to, I am in, I am at, I am next to, I am lost, '
    'yes, no, repeat, cancel, stop, done, '
    'I am in a hurry, I have arrived, my destination, the doors are open.'
)

print(f'Domain prompt: {len(DOMAIN_PROMPT)} chars, {len(DOMAIN_PROMPT.split())} words')


### 2.4 Models and conditions

Eight models total: six Whisper checkpoints covering the full size scale (tiny, base, small, medium, large-v3-turbo, large-v3) and two alternative ASR architectures (Wav2Vec2 CTC, Speech2Text Seq2Seq). All run on the same Colab GPU to eliminate infrastructure confounders.

Each Whisper model is run under both prompt conditions. HF models are run once (they do not accept a prompt) and the result is duplicated across conditions for tabular consistency.

Total reported rows: `(6 Whisper x 2 conditions + 2 HF x 2 duplicated) x 30 phrases = 360 + 120 = 480 rows` from **420 unique inferences**.

In [ ]:
# All models run locally on Colab GPU for a fair comparison on identical hardware.
# Whisper models span the full size scale (tiny -> large-v3) plus two alternative
# ASR architectures (Wav2Vec2 CTC, Speech2Text Seq2Seq) as baselines from other families.
LOCAL_MODELS = [
    {'name': 'tiny', 'group': 'local', 'size': 'tiny'},
    {'name': 'base', 'group': 'local', 'size': 'base'},
    {'name': 'small', 'group': 'local', 'size': 'small'},
    {'name': 'medium', 'group': 'local', 'size': 'medium'},
    {'name': 'large-v3-turbo', 'group': 'local', 'size': 'large-v3-turbo'},
    {'name': 'large-v3', 'group': 'local', 'size': 'large-v3'},
]

# HuggingFace models from other ASR families.
# They do NOT accept a prompt parameter, so the with_prompt condition
# reports the same result as no_prompt for fair tabulation.
HF_MODELS = [
    {'name': 'facebook/wav2vec2-base-960h', 'group': 'hf', 'size': 'wav2vec2-base', 'arch': 'wav2vec2', 'supports_prompt': False},
    {'name': 'facebook/s2t-small-librispeech-asr', 'group': 'hf', 'size': 's2t-small', 'arch': 'speech2text', 'supports_prompt': False},
]

CONDITIONS = ['no_prompt', 'with_prompt']

n_models = len(LOCAL_MODELS) + len(HF_MODELS)
n_total = n_models * len(CONDITIONS) * len(PHRASES)
print(f'Models: {len(LOCAL_MODELS)} Whisper local + {len(HF_MODELS)} HF = {n_models} total')
print(f'Conditions: {CONDITIONS}')
print(f'Total inferences expected: {n_total}')


### 2.5 Output paths

All results are stored under `/content/results/whisper/`. Each model gets its own subdirectory with the per-row CSV.

In [ ]:
RESULTS_PATH = '/content/results/whisper'
PLOTS_DIR = f'{RESULTS_PATH}/comparison_plots'
os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f'Results will be saved to {RESULTS_PATH}')

---
## 3. Reproducibility card

Captures the exact environment (GPU, library versions, git commit, dataset size) so the experiment can be reproduced or audited later.

In [ ]:
def get_git_commit():
    try:
        return subprocess.check_output(
            ['git', 'rev-parse', '--short', 'HEAD'],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return 'N/A'

repro = {
    'experiment': 'whisper_stt_evaluation',
    'date': datetime.datetime.now().isoformat(),
    'python_version': platform.python_version(),
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'git_commit': get_git_commit(),
    'local_models': [m['name'] for m in LOCAL_MODELS],
    'hf_models': [m['name'] for m in HF_MODELS],
    'conditions': CONDITIONS,
    'n_phrases': len(PHRASES),
    'n_keywords': len(DOMAIN_KEYWORDS),
    'domain_prompt_length_chars': len(DOMAIN_PROMPT),
}

repro_path = f'{RESULTS_PATH}/whisper_reproducibility.json'
with open(repro_path, 'w') as f:
    json.dump(repro, f, indent=2)
print(f'Reproducibility card saved to {repro_path}')
print(json.dumps(repro, indent=2))


---
## 4. Helper functions

Text normalization, metric computation, and inference wrappers. Splitting helpers across small cells keeps each function focused and easy to inspect.

### 4.1 Text normalization

Two normalizers:

- `normalize_text` strips incidental punctuation (commas, semicolons, trailing dots) but preserves decimal dots in numbers like `0.3` and `1.4`. Used for `Sentence Accuracy` and as input to WER/CER.
- `normalize_for_dka` additionally rewrites spelled-out numbers (`zero point three`) into their numeric form (`0.3`). This avoids penalizing models that dictate numbers as words.

In [ ]:
_PUNCT_RE = re.compile(r"[,;:!?\"\(\)]")
_TRAIL_DOT_RE = re.compile(r"(?<!\d)\.|\.(?!\d)")
_WS_RE = re.compile(r"\s+")

_SPOKEN_NUMS = {
    'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
    'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
}

def normalize_text(text):
    """Lowercase, strip incidental punctuation, collapse whitespace.
    Preserves dots inside decimal numbers (0.3, 1.4).
    """
    if text is None:
        return ''
    t = str(text).lower().strip()
    t = _PUNCT_RE.sub(' ', t)
    t = _TRAIL_DOT_RE.sub('', t)
    t = _WS_RE.sub(' ', t).strip()
    return t

def normalize_for_dka(text):
    """Same as normalize_text but additionally rewrites 'zero point three' -> '0.3'.
    Used for Domain Keyword Accuracy comparisons.
    """
    t = normalize_text(text)
    pattern = re.compile(r'\b(' + '|'.join(_SPOKEN_NUMS) + r')\s+point\s+(' + '|'.join(_SPOKEN_NUMS) + r')\b')
    def repl(m):
        return f"{_SPOKEN_NUMS[m.group(1)]}.{_SPOKEN_NUMS[m.group(2)]}"
    return pattern.sub(repl, t)

### 4.2 Per-phrase metrics

WER and CER come from `jiwer` and operate on normalized text. `Sentence Accuracy` is a strict equality check after normalization. `DKA` returns `None` when the ground truth contains no tracked keywords (so phrases without domain content do not pull the average up or down).

In [ ]:
def safe_wer(gt, pred):
    if not gt:
        return 0.0
    return float(wer_fn(normalize_text(gt), normalize_text(pred)))

def safe_cer(gt, pred):
    if not gt:
        return 0.0
    return float(cer_fn(normalize_text(gt), normalize_text(pred)))

def compute_sentence_accuracy(prediction, ground_truth):
    """1.0 if normalized prediction equals normalized ground truth; 0.0 otherwise."""
    return 1.0 if normalize_text(prediction) == normalize_text(ground_truth) else 0.0

def compute_dka(prediction, ground_truth, keywords):
    """Fraction of ground-truth domain keywords that also appear in the prediction.
    Returns None if no tracked keyword is present in the ground truth.
    """
    pred_norm = normalize_for_dka(prediction)
    gt_norm = normalize_for_dka(ground_truth)
    gt_keywords = [kw for kw in keywords if kw in gt_norm]
    if not gt_keywords:
        return None
    found = sum(1 for kw in gt_keywords if kw in pred_norm)
    return found / len(gt_keywords)

### 4.3 Aggregation helper

Reduces a list of per-phrase row dicts to a single dict of mean metrics. `DKA` is averaged only over phrases where it is defined (i.e. that contain at least one tracked keyword).

In [ ]:
def aggregate_results(rows):
    df = pd.DataFrame(rows)
    dka_vals = df['dka'].dropna()
    return {
        'wer_pct': round(df['wer'].mean() * 100, 2),
        'cer_pct': round(df['cer'].mean() * 100, 2),
        'sentence_accuracy_pct': round(df['sentence_accuracy'].mean() * 100, 2),
        'dka_pct': round(dka_vals.mean() * 100, 2) if len(dka_vals) > 0 else None,
        'avg_latency_s': round(df['latency_s'].mean(), 3),
        'n_phrases': len(df),
    }

def safe_model_name(name):
    return name.replace('/', '_').replace('-', '_')

### 4.4 Transcription wrappers

Two thin wrappers, one per backend. Both return `(text, latency_seconds)` so the inference loop stays identical regardless of backend.

In [ ]:
def transcribe_local(model, audio_path, prompt=None):
    """Transcribe a single audio with a local openai-whisper model. Returns (text, latency_s)."""
    kwargs = {'language': 'en'}
    if prompt:
        kwargs['initial_prompt'] = prompt
    t0 = time.time()
    result = model.transcribe(audio_path, **kwargs)
    latency = time.time() - t0
    return result['text'].strip(), latency

def transcribe_hf(model, processor, arch, audio_path):
    """Transcribe with a HuggingFace ASR model (wav2vec2 or speech2text).
    These architectures do not accept a prompt parameter. Returns (text, latency_s).
    """
    import librosa
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        if arch == 'wav2vec2':
            logits = model(inputs.input_values).logits
            predicted_ids = torch.argmax(logits, dim=-1)
            text = processor.batch_decode(predicted_ids)[0]
        elif arch == 'speech2text':
            generated_ids = model.generate(
                inputs.input_features,
                attention_mask=inputs.get('attention_mask'),
            )
            text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        else:
            raise ValueError(f'Unknown HF arch: {arch}')
    latency = time.time() - t0
    return text.strip(), latency


### 4.5 Sanity checks

Tiny smoke test on the helpers before running the full experiment, so any normalization regression fails immediately rather than at aggregation time.

In [ ]:
assert normalize_text('Hello, world.') == 'hello world'
assert normalize_text('Classroom 0.3.') == 'classroom 0.3'
assert normalize_text('Building C.D.') == 'building cd'  # acronyms with dots collapse
assert normalize_text('U.S.A.') == 'usa'
assert compute_sentence_accuracy('Hello, world.', 'hello world') == 1.0
assert compute_sentence_accuracy('Building C.D.', 'building cd') == 1.0
assert compute_dka('I am in classroom 0.3', 'I am in classroom 0.3 building G', DOMAIN_KEYWORDS) == 0.5
assert normalize_for_dka('zero point three') == '0.3'
print('Sanity checks passed.')


---
## 5. Run experiments

Two independent loops, one per inference backend. Each iterates over `(model, condition, phrase)` and stores per-row results plus per-model aggregates.

### 5.1 Local Whisper models on Colab GPU

Six local Whisper models (tiny, base, small, medium, large-v3-turbo, large-v3) x two conditions x 30 phrases = **360 inferences**. Models are released from VRAM between iterations to avoid out-of-memory errors.

In [ ]:
all_summaries = []

for meta in LOCAL_MODELS:
    model_name = meta['name']
    safe = safe_model_name(f'whisper-{model_name}')
    model_dir = f'{RESULTS_PATH}/{safe}'
    os.makedirs(model_dir, exist_ok=True)

    print()
    print('=' * 60)
    print(f'LOCAL: whisper-{model_name}')
    print('=' * 60)

    t0 = time.time()
    print('  Loading model...')
    local_model = whisper.load_model(model_name, device=DEVICE)
    print(f'  Loaded in {time.time() - t0:.1f}s')

    rows_all = []
    for condition in CONDITIONS:
        prompt_to_use = DOMAIN_PROMPT if condition == 'with_prompt' else None
        print(f'  -> condition: {condition}')
        rows = []
        for phrase in PHRASES:
            audio_path = os.path.join(AUDIO_DIR, phrase['audio'])
            try:
                pred, latency = transcribe_local(local_model, audio_path, prompt_to_use)
            except Exception as e:
                print(f'     [FAIL] {phrase["id"]}: {e}')
                pred, latency = '', 0.0
            gt = phrase['ground_truth']
            row = {
                'model': f'whisper-{model_name}',
                'group': 'local',
                'condition': condition,
                'phrase_id': phrase['id'],
                'category': phrase['category'],
                'ground_truth': gt,
                'prediction': pred,
                'wer': safe_wer(gt, pred),
                'cer': safe_cer(gt, pred),
                'sentence_accuracy': compute_sentence_accuracy(pred, gt),
                'dka': compute_dka(pred, gt, DOMAIN_KEYWORDS),
                'latency_s': round(latency, 4),
            }
            rows.append(row)
            rows_all.append(row)

        agg = aggregate_results(rows)
        agg.update({
            'model': f'whisper-{model_name}', 'group': 'local', 'size': meta['size'],
            'condition': condition, 'failed': 0,
        })
        all_summaries.append(agg)
        print(f'     WER={agg["wer_pct"]:.2f}% CER={agg["cer_pct"]:.2f}% SentAcc={agg["sentence_accuracy_pct"]:.2f}% DKA={agg["dka_pct"]}% Lat={agg["avg_latency_s"]:.3f}s')

    pd.DataFrame(rows_all).to_csv(f'{model_dir}/results.csv', index=False)
    print(f'  Saved: {model_dir}/results.csv ({len(rows_all)} rows)')

    del local_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
print(f'Local experiment complete. {len(LOCAL_MODELS) * len(CONDITIONS) * len(PHRASES)} inferences.')

### 5.2 HuggingFace ASR models (Wav2Vec2 and Speech2Text)

Two models from other ASR families to contrast against the Whisper line.

- `facebook/wav2vec2-base-960h` - CTC-based ASR, English-only, trained on 960h LibriSpeech
- `facebook/s2t-small-librispeech-asr` - Sequence-to-sequence ASR, English-only

**Important:** these architectures do not accept a prompt parameter. For consistency with the rest of the experiment, the same transcription is reported under both `no_prompt` and `with_prompt` conditions, which makes the prompt-impact delta exactly zero (a valid finding: prompt steering does not generalize beyond Whisper).

In [ ]:
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    Speech2TextForConditionalGeneration, Speech2TextProcessor,
)

for meta in HF_MODELS:
    model_name = meta['name']
    safe = safe_model_name(model_name)
    model_dir = f'{RESULTS_PATH}/{safe}'
    os.makedirs(model_dir, exist_ok=True)

    print()
    print('=' * 60)
    print(f'HF: {model_name}')
    print('=' * 60)

    t0 = time.time()
    print('  Loading model...')
    if meta['arch'] == 'wav2vec2':
        processor = Wav2Vec2Processor.from_pretrained(model_name)
        hf_model = Wav2Vec2ForCTC.from_pretrained(model_name).to(DEVICE).eval()
    elif meta['arch'] == 'speech2text':
        processor = Speech2TextProcessor.from_pretrained(model_name)
        hf_model = Speech2TextForConditionalGeneration.from_pretrained(model_name).to(DEVICE).eval()
    else:
        raise ValueError(f'Unknown arch: {meta["arch"]}')
    print(f'  Loaded in {time.time() - t0:.1f}s')

    rows_all = []
    rows_no_prompt = []
    failed = 0
    print('  -> running inference (single pass, prompt not supported)')
    for phrase in PHRASES:
        audio_path = os.path.join(AUDIO_DIR, phrase['audio'])
        try:
            pred, latency = transcribe_hf(hf_model, processor, meta['arch'], audio_path)
        except Exception as e:
            print(f'     [FAIL] {phrase["id"]}: {e}')
            pred, latency = '', 0.0
            failed += 1
        gt = phrase['ground_truth']
        # Same prediction is reported in both conditions (HF models ignore prompt)
        for condition in CONDITIONS:
            row = {
                'model': model_name,
                'group': 'hf',
                'condition': condition,
                'phrase_id': phrase['id'],
                'category': phrase['category'],
                'ground_truth': gt,
                'prediction': pred,
                'wer': safe_wer(gt, pred),
                'cer': safe_cer(gt, pred),
                'sentence_accuracy': compute_sentence_accuracy(pred, gt),
                'dka': compute_dka(pred, gt, DOMAIN_KEYWORDS),
                'latency_s': round(latency, 4),
            }
            rows_all.append(row)
            if condition == 'no_prompt':
                rows_no_prompt.append(row)

    # Aggregate (use no_prompt rows since both conditions are identical)
    for condition in CONDITIONS:
        agg = aggregate_results(rows_no_prompt)
        agg.update({
            'model': model_name, 'group': 'hf', 'size': meta['size'],
            'condition': condition, 'failed': failed,
        })
        all_summaries.append(agg)
        line = '     [{}] WER={:.2f}% CER={:.2f}% SentAcc={:.2f}% DKA={}% Lat={:.3f}s'.format(
            condition, agg['wer_pct'], agg['cer_pct'], agg['sentence_accuracy_pct'], agg['dka_pct'], agg['avg_latency_s'])
        print(line)

    pd.DataFrame(rows_all).to_csv(f'{model_dir}/results.csv', index=False)
    print(f'  Saved: {model_dir}/results.csv ({len(rows_all)} rows)')

    del hf_model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
print(f'HF experiment complete. {len(HF_MODELS)} models x {len(PHRASES)} phrases (single pass) reported in {len(CONDITIONS)} conditions.')


---
## 6. Build summary tables

Two tables are produced from the in-memory `all_summaries` list and saved alongside the per-model CSVs.

### 6.1 Long table (one row per model x condition)

The natural format for downstream pivoting and plotting.

In [ ]:
summary_df = pd.DataFrame(all_summaries)
cols_order = [
    'model', 'group', 'size', 'condition',
    'wer_pct', 'cer_pct', 'sentence_accuracy_pct', 'dka_pct',
    'avg_latency_s', 'n_phrases', 'failed',
]
summary_df = summary_df[[c for c in cols_order if c in summary_df.columns]]
summary_df = summary_df.sort_values(['group', 'model', 'condition']).reset_index(drop=True)

summary_df.to_csv(f'{RESULTS_PATH}/all_models_summary_long.csv', index=False)
print(f'Saved: {RESULTS_PATH}/all_models_summary_long.csv')
print()
print(summary_df.to_string(index=False))

### 6.2 Wide table with prompt-impact deltas

One row per model with `no_prompt` and `with_prompt` side by side, plus the deltas. This is the table you would put in the thesis to argue the prompt's contribution.

In [ ]:
wide_rows = []
for model_name in summary_df['model'].unique():
    sub = summary_df[summary_df['model'] == model_name]
    np_ = sub[sub['condition'] == 'no_prompt']
    wp = sub[sub['condition'] == 'with_prompt']
    if np_.empty or wp.empty:
        continue
    np_, wp = np_.iloc[0], wp.iloc[0]
    wide_rows.append({
        'model': model_name,
        'group': np_['group'],
        'size': np_['size'],
        'wer_no_prompt': np_['wer_pct'],
        'wer_with_prompt': wp['wer_pct'],
        'wer_delta': round(np_['wer_pct'] - wp['wer_pct'], 2),
        'cer_no_prompt': np_['cer_pct'],
        'cer_with_prompt': wp['cer_pct'],
        'cer_delta': round(np_['cer_pct'] - wp['cer_pct'], 2),
        'sentence_acc_no_prompt': np_['sentence_accuracy_pct'],
        'sentence_acc_with_prompt': wp['sentence_accuracy_pct'],
        'dka_no_prompt': np_['dka_pct'],
        'dka_with_prompt': wp['dka_pct'],
        'dka_delta': round((wp['dka_pct'] or 0) - (np_['dka_pct'] or 0), 2),
        'latency_no_prompt_s': np_['avg_latency_s'],
        'latency_with_prompt_s': wp['avg_latency_s'],
    })
wide_df = pd.DataFrame(wide_rows)
wide_df.to_csv(f'{RESULTS_PATH}/all_models_summary_wide.csv', index=False)
print(f'Saved: {RESULTS_PATH}/all_models_summary_wide.csv')
print()
print(wide_df.to_string(index=False))

### 6.3 Per-utterance comparison table

For each utterance, shows the ground truth alongside the transcription produced by every model x condition combination. This is the qualitative view that lets you inspect *what each model heard* phrase by phrase, instead of just the aggregated metrics.

Saved as `per_utterance_predictions.csv`.

In [ ]:
# Build a wide per-utterance table: rows = utterances, columns = each model x condition prediction
all_rows = []
for safe_dir in os.listdir(RESULTS_PATH):
    p = os.path.join(RESULTS_PATH, safe_dir, 'results.csv')
    if os.path.isfile(p):
        all_rows.append(pd.read_csv(p))

if not all_rows:
    print('No per-model CSVs found. Skipping per-utterance view.')
else:
    full_df = pd.concat(all_rows, ignore_index=True)

    # Pivot: rows = phrase_id, one column per (model, condition)
    full_df['col'] = full_df['model'] + ' [' + full_df['condition'] + ']'
    pivot = full_df.pivot_table(
        index=['phrase_id', 'category', 'ground_truth'],
        columns='col',
        values='prediction',
        aggfunc='first',
    ).reset_index()

    pivot.to_csv(f'{RESULTS_PATH}/per_utterance_predictions.csv', index=False)
    print(f'Saved: {RESULTS_PATH}/per_utterance_predictions.csv')
    print(f'Shape: {pivot.shape[0]} utterances x {pivot.shape[1] - 3} model-condition columns')
    print()

    # Show each utterance with its predictions in a vertical, readable format
    pred_cols = [c for c in pivot.columns if c not in {"phrase_id", "category", "ground_truth"}]
    for _, row in pivot.iterrows():
        print('=' * 80)
        print(f'{row["phrase_id"]}  [{row["category"]}]')
        print(f'  GROUND TRUTH: {row["ground_truth"]}')
        print('  ---')
        for col in pred_cols:
            pred = row[col]
            if pd.isna(pred):
                pred = "(missing)"
            print(f'  {col}:')
            print(f'    {pred}')
        print()


### 6.4 Prompt Impact Score (PIS)

Explicit naming of the prompt-impact deltas, used as a model characterization metric:

- **PIS_DKA = DKA_with_prompt - DKA_no_prompt** (higher = the model relies more on the prompt to recover domain vocabulary)
- **PIS_WER = WER_no_prompt - WER_with_prompt** (positive = the prompt reduces transcription errors)

A high PIS indicates a model that benefits significantly from domain context. Whisper models with low PIS are more robust without prompting. HF models have PIS = 0 by design (they do not accept a prompt parameter), which is itself a meaningful finding.

In [ ]:
pis_df = wide_df[[
    'model', 'group', 'size',
    'dka_no_prompt', 'dka_with_prompt', 'dka_delta',
    'wer_no_prompt', 'wer_with_prompt', 'wer_delta',
]].copy()
pis_df = pis_df.rename(columns={'dka_delta': 'pis_dka', 'wer_delta': 'pis_wer'})
pis_df = pis_df.sort_values('pis_dka', ascending=False).reset_index(drop=True)
pis_df.to_csv(f'{RESULTS_PATH}/prompt_impact_score.csv', index=False)
print(f'Saved: {RESULTS_PATH}/prompt_impact_score.csv')
print()
print('=== Prompt Impact Score (sorted by PIS_DKA) ===')
print(pis_df.to_string(index=False))


---
## 7. Comparison plots

All bar charts use the same scheme: grey for `no_prompt`, green for `with_prompt`. One plot per metric so each figure can stand alone in the thesis.

### 7.1 Shared plotting helper

Centralizes the styling so all metric plots look consistent.

In [ ]:
def grouped_bar(metric_col, title, ylabel, filename, lower_is_better=False):
    pivot = summary_df.pivot(index='model', columns='condition', values=metric_col)
    pivot = pivot.reindex(columns=['no_prompt', 'with_prompt'])
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(pivot))
    width = 0.35
    ax.bar(x - width / 2, pivot['no_prompt'], width, label='no prompt', color='#90A4AE')
    ax.bar(x + width / 2, pivot['with_prompt'], width, label='with prompt', color='#4CAF50')
    for i, model in enumerate(pivot.index):
        for j, cond in enumerate(['no_prompt', 'with_prompt']):
            v = pivot.loc[model, cond]
            if pd.notna(v):
                ax.text(i + (j - 0.5) * width, v + 0.5, f'{v:.1f}', ha='center', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    if lower_is_better:
        ax.set_ylim(0, max(pivot.max().max() * 1.2, 5))
    else:
        ax.set_ylim(0, 110)
    plt.tight_layout()
    out = f'{PLOTS_DIR}/{filename}'
    plt.savefig(out, dpi=120)
    plt.close()
    print(f'Saved: {out}')

### 7.2 Word and Character Error Rates (lower is better)

In [ ]:
grouped_bar('wer_pct', 'Word Error Rate by Model and Condition', 'WER (%)', 'cmp_wer.png', lower_is_better=True)
grouped_bar('cer_pct', 'Character Error Rate by Model and Condition', 'CER (%)', 'cmp_cer.png', lower_is_better=True)

### 7.3 Sentence Accuracy and Domain Keyword Accuracy (higher is better)

In [ ]:
grouped_bar('sentence_accuracy_pct', 'Sentence Accuracy by Model and Condition', 'Sentence Accuracy (%)', 'cmp_sentence_accuracy.png')
grouped_bar('dka_pct', 'Domain Keyword Accuracy by Model and Condition', 'DKA (%)', 'cmp_dka.png')

### 7.4 Latency by model

Latency is measured on the same Colab GPU for every model, so the comparison is direct and fair. The plot shows average latency per inference for each model under both prompt conditions.

In [ ]:
# Single latency plot - all models on the same axis since they share GPU infrastructure.
pivot = summary_df.pivot(index='model', columns='condition', values='avg_latency_s')
pivot = pivot.reindex(columns=['no_prompt', 'with_prompt'])

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(pivot))
w = 0.35
ax.bar(x - w / 2, pivot['no_prompt'], w, label='no prompt', color='#90A4AE')
ax.bar(x + w / 2, pivot['with_prompt'], w, label='with prompt', color='#4CAF50')
ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Avg latency per inference (s)')
ax.set_title('Inference latency by model and condition (Colab GPU)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = f'{PLOTS_DIR}/cmp_latency.png'
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved: {out}')


### 7.5 DKA by phrase category

Aggregates per-row DKA across all models to reveal where the domain prompt has the largest impact (typically the `hard_domain` and `initial_location` categories with room codes).

In [ ]:
all_rows = []
for safe_dir in os.listdir(RESULTS_PATH):
    p = os.path.join(RESULTS_PATH, safe_dir, 'results.csv')
    if os.path.isfile(p):
        all_rows.append(pd.read_csv(p))

if all_rows:
    full_df = pd.concat(all_rows, ignore_index=True)
    cat_pivot = full_df.groupby(['category', 'condition'])['dka'].mean().unstack() * 100
    cat_pivot = cat_pivot.reindex(columns=['no_prompt', 'with_prompt'])
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(cat_pivot))
    w = 0.35
    ax.bar(x - w / 2, cat_pivot['no_prompt'], w, label='no prompt', color='#90A4AE')
    ax.bar(x + w / 2, cat_pivot['with_prompt'], w, label='with prompt', color='#4CAF50')
    ax.set_xticks(x)
    ax.set_xticklabels(cat_pivot.index, rotation=20, ha='right')
    ax.set_ylabel('DKA (%)')
    ax.set_title('DKA by phrase category (averaged across all models)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 110)
    plt.tight_layout()
    out = f'{PLOTS_DIR}/cmp_dka_by_category.png'
    plt.savefig(out, dpi=120)
    plt.close()
    print(f'Saved: {out}')
else:
    print('No per-model CSVs found - skipping category breakdown.')

### 7.6 Per-category breakdown by model (heatmaps)

Heatmaps that answer "in which category does each model fail most?". Rows are models, columns are categories. The cell value is the metric averaged over the with_prompt condition (which is the production-equivalent setting). Darker red cells mark weak combinations; greener cells mark strong ones.

In [ ]:
# Reload full per-row data (already loaded for 7.5 but re-do to be self-contained)
all_rows = []
for safe_dir in os.listdir(RESULTS_PATH):
    p = os.path.join(RESULTS_PATH, safe_dir, 'results.csv')
    if os.path.isfile(p):
        all_rows.append(pd.read_csv(p))
full_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

def category_heatmap(metric, title, filename, vmin=0, vmax=100, cmap='RdYlGn'):
    if full_df.empty:
        print('No per-model CSVs available for heatmap.')
        return
    sub = full_df[full_df['condition'] == 'with_prompt'].copy()
    if metric == 'dka':
        pivot = sub.groupby(['model', 'category'])[metric].apply(lambda s: s.dropna().mean()).unstack() * 100
    else:
        pivot = sub.groupby(['model', 'category'])[metric].mean().unstack() * 100
    pivot = pivot.reindex(['initial_location', 'destination', 'navigation_command', 'confirmation', 'hard_domain', 'edge_case'], axis=1)
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(pivot.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=20, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if pd.notna(v):
                ax.text(j, i, f'{v:.1f}', ha='center', va='center', color='black', fontsize=8)
    plt.colorbar(im, ax=ax, label='%')
    ax.set_title(title)
    plt.tight_layout()
    out = f'{PLOTS_DIR}/{filename}'
    plt.savefig(out, dpi=120)
    plt.close()
    print(f'Saved: {out}')

# WER: lower is better -> use reversed colormap (red=high error)
category_heatmap('wer', 'WER (%) by model and category [with_prompt]', 'cmp_wer_by_model_category.png', vmin=0, vmax=30, cmap='RdYlGn_r')
# Sentence Accuracy: higher is better
category_heatmap('sentence_accuracy', 'Sentence Accuracy (%) by model and category [with_prompt]', 'cmp_sentacc_by_model_category.png', vmin=0, vmax=100, cmap='RdYlGn')
# DKA: higher is better
category_heatmap('dka', 'DKA (%) by model and category [with_prompt]', 'cmp_dka_by_model_category.png', vmin=0, vmax=100, cmap='RdYlGn')


### 7.7 Pareto trade-off: quality vs speed

Scatter plot of every (model, condition) pair on the DKA-vs-Latency plane. Models in the **top-left corner** (high DKA, low latency) dominate the rest. The shape encodes the prompt condition (circle = no_prompt, square = with_prompt) and the colour encodes the model family (Whisper in green, HuggingFace ASR in orange). Useful for justifying the model choice for production deployment.

In [ ]:
from matplotlib.lines import Line2D

colors = {'local': '#4CAF50', 'hf': '#FF9800'}
markers = {'no_prompt': 'o', 'with_prompt': 's'}

fig, ax = plt.subplots(figsize=(11, 6))

for _, row in summary_df.iterrows():
    if row['dka_pct'] is None or pd.isna(row['dka_pct']):
        continue
    ax.scatter(
        row['avg_latency_s'], row['dka_pct'],
        s=140, c=colors.get(row['group'], 'gray'),
        marker=markers.get(row['condition'], 'o'),
        edgecolors='black', linewidths=0.6, alpha=0.85,
    )
    label = row['model'].split('/')[-1]
    ax.annotate(
        f'{label}',
        xy=(row['avg_latency_s'], row['dka_pct']),
        xytext=(6, 4), textcoords='offset points', fontsize=7,
    )

ax.set_xlabel('Avg latency per inference (s)')
ax.set_ylabel('DKA (%)')
ax.set_title('Pareto trade-off: Quality (DKA) vs Speed (Latency)')
ax.grid(alpha=0.3)
ax.set_ylim(0, 110)

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['local'], markeredgecolor='black', label='Whisper local', markersize=10),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['hf'], markeredgecolor='black', label='HuggingFace', markersize=10),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markeredgecolor='black', label='no_prompt', markersize=10),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markeredgecolor='black', label='with_prompt', markersize=10),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
out = f'{PLOTS_DIR}/cmp_pareto_dka_vs_latency.png'
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved: {out}')


### 7.8 Utterance-level difficulty

Aggregates WER, Sentence Accuracy and DKA across all models and both conditions per utterance. The output ranks the utterances from hardest to easiest, identifying the **killer phrases** that separate strong models from weak ones - the natural source of qualitative examples for the thesis Discussion section.

In [ ]:
if full_df.empty:
    print('No per-model CSVs available for utterance difficulty analysis.')
else:
    diff_rows = []
    for phrase_id in full_df['phrase_id'].unique():
        sub = full_df[full_df['phrase_id'] == phrase_id]
        diff_rows.append({
            'phrase_id': phrase_id,
            'category': sub['category'].iloc[0],
            'ground_truth': sub['ground_truth'].iloc[0],
            'mean_wer_pct': round(sub['wer'].mean() * 100, 2),
            'max_wer_pct': round(sub['wer'].max() * 100, 2),
            'mean_sentence_accuracy_pct': round(sub['sentence_accuracy'].mean() * 100, 2),
            'mean_dka_pct': round(sub['dka'].dropna().mean() * 100, 2) if sub['dka'].dropna().size else None,
            'n_runs': len(sub),
        })

    diff_df = pd.DataFrame(diff_rows)
    diff_df = diff_df.sort_values('mean_wer_pct', ascending=False).reset_index(drop=True)
    diff_df.to_csv(f'{RESULTS_PATH}/utterance_difficulty.csv', index=False)
    print(f'Saved: {RESULTS_PATH}/utterance_difficulty.csv')
    print()
    print('=== Utterance difficulty (hardest first) ===')
    print(diff_df.to_string(index=False))


---
## 8. Final ranking and downloads

### 8.1 Ranking by DKA in the *with-prompt* condition

DKA in the with-prompt condition is the metric that best reflects real-world utility for the navigation domain: it scores how well each model preserves the vocabulary required by the downstream system, when given the domain context that production already provides.

In [ ]:
ranking_cols = ['model', 'group', 'wer_pct', 'cer_pct', 'sentence_accuracy_pct', 'dka_pct', 'avg_latency_s']
wp = summary_df[summary_df['condition'] == 'with_prompt'].sort_values('dka_pct', ascending=False)
print('FINAL RANKING (with prompt, sorted by DKA)')
print('=' * 70)
print(wp[ranking_cols].to_string(index=False))

### 8.2 Prompt impact summary

`wer_delta = wer_no_prompt - wer_with_prompt` (positive = prompt reduced WER).

`dka_delta = dka_with_prompt - dka_no_prompt` (positive = prompt improved DKA).

In [ ]:
print('PROMPT IMPACT (positive = prompt helped)')
print('=' * 70)
print(wide_df[['model', 'wer_delta', 'dka_delta']].to_string(index=False))

### 8.3 Output file inventory

In [ ]:
print('Output files:')
print(f'  {RESULTS_PATH}/whisper_reproducibility.json')
print(f'  {RESULTS_PATH}/all_models_summary_long.csv')
print(f'  {RESULTS_PATH}/all_models_summary_wide.csv')
print(f'  {RESULTS_PATH}/<model>/results.csv  (per-model row-level details)')
print(f'  {PLOTS_DIR}/*.png')

### 8.4 Download the full results folder

Run the next cell once the experiment has finished to bundle everything into a single `.zip` and download it to your local machine.

In [ ]:
from google.colab import files

!zip -r /content/whisper_results.zip /content/results/whisper > /dev/null
files.download('/content/whisper_results.zip')